**IMPORTANT**
This is dummy data! model works


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt


from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericStocks, MaterialIntensities

from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.appliances.preprocessing.main import appliances_preprocessing

import prism

from pathlib import Path
from importlib.resources import files

prism.unit_registry.load_definitions(files("imagematerials") / "units.txt")
ureg = prism.unit_registry


In [ ]:
scenario_list = {"SSP2_baseline":("SSP2_baseline", None)}

In [ ]:
climate_scen = "SSP2_baseline"
image_data = Path("..", "data", "raw", "image", climate_scen)

In [ ]:
time_start = 1971
complete_timeline = prism.Timeline(1971, 2100, 1)
simulation_timeline = prism.Timeline(1971, 2100, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", climate_scen)
    circular_economy_scenario_dirs = None

    appliances_sector = get_preprocessing_data("appliances", 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs) 

    factory = ModelFactory(
    [appliances_sector], complete_timeline
    ).add(GenericStocks, ["appliances"]
    ).add(MaterialIntensities, "appliances",
)
    model = factory.finish()

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    print(f"Finished {scen_id}")

In [ ]:
inflow_materials = model.appliances.get("inflow_materials").to_array().sum(["Region"]).pint.to("Mt")
stocks_materials = model.appliances.get("stock_by_cohort_materials").sum(["Region"]).pint.to("Mt")
outflow_materials = model.appliances.get("outflow_by_cohort_materials").to_array().sum(["Region"]).pint.to("Mt")

In [ ]:
# Inflow of materials for appliances, stacked by material

start_year = 2025


fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=False)
ax1, ax2, ax3, ax4, ax5, ax6 = axes.flatten()

material_names = inflow_materials.material.values
ax1.stackplot(
    inflow_materials.time.loc[start_year:],
    [inflow_materials.sum("Type").sel(material=m).loc[start_year:] for m in material_names],
    labels=material_names
)

material_names = inflow_materials.material.values
ax2.stackplot(
    stocks_materials.time.loc[start_year:],
    [stocks_materials.sum("Type").sel(material=m).loc[start_year:] for m in material_names],
    labels=material_names
)

material_names = outflow_materials.material.values
ax3.stackplot(
    outflow_materials.time.loc[start_year:],
    [outflow_materials.sum("Type").sel(material=m).loc[start_year:] for m in material_names],
    labels=material_names
)

typenames = inflow_materials.Type.values
ax4.stackplot(
    inflow_materials.time.loc[start_year:],
    [inflow_materials.sum("material").sel(Type=m).loc[start_year:] for m in typenames],
    labels=typenames
)

typenames = inflow_materials.Type.values
ax5.stackplot(
    stocks_materials.time.loc[start_year:],
    [stocks_materials.sum("material").sel(Type=m).loc[start_year:] for m in typenames],
    labels=typenames
)

material_names = outflow_materials.material.values
ax6.stackplot(
    outflow_materials.time.loc[start_year:],
    [outflow_materials.sum("material").sel(Type=m).loc[start_year:] for m in typenames],
    labels=material_names
)

for ax in axes.flatten():
    ax.set_ylabel('Year')
    ax.set_xlabel('Mt')


# Combined legend: material names (from the materials row) + type names (from the types row)
handles_materials, labels_materials = ax1.get_legend_handles_labels()
handles_types, labels_types = ax4.get_legend_handles_labels()
fig.legend(
    handles_materials + handles_types,
    labels_materials + labels_types,
    loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False
)

plt.show()